# H SC 790 · Lab 8 · Break a "De-identified" Release, Then Repair It

**Week 9.** Due Thursday, March 25, 11:59 PM · 30 points.

**What earns full credit**
- Runs end to end from a fresh runtime
- Risk measured before and after each step
- Utility cost shown with the same analysis
- AI-use note included

**Rules.** Everything here is synthetic. Never load protected health information or confidential employer data into Colab or GitHub. The lab is finished when it runs top to bottom from a fresh runtime (Runtime → Restart session and run all).

The data dictionary for this lab is the README in the lab's folder of the course repository.

Cells marked `# YOUR CODE HERE` and *Your answer here.* are yours to complete. Add cells freely.

In [ ]:
# Setup: run this cell first, every time.
import os, json, urllib.request
import numpy as np
import pandas as pd

# The course data, read straight from the course repository. Never from your own machine.
COURSE_URL = os.environ.get("HSC790_DATA", "https://raw.githubusercontent.com/OWNER/hsc790-spr27/main/data/")

def read_text(rel):
    """Text of one file in the course data folder, for example read_text('SNAPSHOT.txt')."""
    return urllib.request.urlopen(COURSE_URL + rel).read().decode()

print("Data snapshot:", read_text("SNAPSHOT.txt").strip())

A hospital discharge file for Foothills residents was released with names and addresses removed. Measure how identifiable it is, attack it with a public-style roster, protect it, and measure what the protection costs.

Everything here is synthetic: the release, the roster, and every person in them. That is why you may run this attack. You cannot judge the risk of a release you are responsible for without knowing how one is broken.

In [ ]:
rel = pd.read_csv(COURSE_URL + "lab08/discharge_release.csv", dtype={"zip": str})
roster = pd.read_csv(COURSE_URL + "lab08/voter_roster.csv", dtype={"zip": str})
popn = pd.read_csv(COURSE_URL + "lab08/population_by_zip_age_sex.csv", dtype={"zip": str})
ref = pd.read_csv(COURSE_URL + "lab08/reference_rates.csv", dtype={"zip": str})
np.random.seed(790)
print(len(rel), "stays;", len(roster), "voters")
rel.head(3)

## 1. Measure the release as given

Choose the quasi-identifiers. Report the share of unique records, the smallest class, and the distribution of k.

In [ ]:
# --- risk: class size k for every record (slide) ---
qi = ['zip', 'birth_date', 'sex']
rel['k'] = rel.groupby(qi)['sex'].transform('size')
print('unique:', (rel.k == 1).mean(),
      ' smallest class:', rel.k.min())

# YOUR CODE HERE

## 2. Run the linkage attack

Join the release to the roster on the quasi-identifiers. How many records get exactly one name? A record that matches exactly one voter has, for an attacker, a name and an address attached to a diagnosis.

In [ ]:
# YOUR CODE HERE

## 3. Apply Safe Harbor

Year of birth only, three-digit ZIP code with the population rule (a ZIP3 area with 20,000 people or fewer becomes `000`), ages over 89 grouped, and dates reduced to the year. Re-measure and re-attack, generalizing the roster the same way an attacker would.

In [ ]:
# --- protect: generalize the quasi-identifiers (slide) ---
safe = rel.assign(
    zip3=rel.zip.str[:3],
    birth_year=pd.to_datetime(rel.birth_date).dt.year,
).drop(columns=['zip', 'birth_date'])

# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

## 4. Go further

Reach k of at least 5 by generalizing and suppressing. Then check each class for homogeneity: a class in which everyone has the same sensitive value discloses that value even though nobody is singled out. Fix what you find.

In [ ]:
# YOUR CODE HERE

## 5. Measure the cost

Run the reference analysis (discharge rate per 1,000 by ZIP code and age group, in `reference_rates.csv`) on the original and on each protected version. What can no longer be answered?

In [ ]:
# YOUR CODE HERE

## 6. Try a noisy table

Instead of releasing records, release the table of counts by ZIP code and age group with Laplace noise, at three values of epsilon. Compare the error for large and small cells.

In [ ]:
# --- a differentially private table of counts (slide) ---
def dp_counts(counts, eps):
    noise = np.random.laplace(0, 1 / eps, len(counts))
    return (counts + noise).clip(lower=0).round()

# YOUR CODE HERE

## Interpretation

Cite your own output in each answer.

**1. How exposed was the original?** Share unique, and share that received exactly one name in the attack.

*Your answer here.*

**2. Was Safe Harbor enough?** What remained unique afterward, and why those records?

*Your answer here.*

**3. What did protection cost?** Name one analysis the protected data can no longer support.

*Your answer here.*

**4. Who should approve a release?** Name the role, and the three numbers they should see first.

*Your answer here.*

## AI-use note (required)

Say which AI tools you used, for what, and one thing you checked or corrected in their output. "None" is an acceptable answer.

*Your note here.*

## Submit

1. Runtime → Restart session and run all. Fix anything that fails.
2. File → Save a copy in GitHub, into `labs/lab08/` of your course repository.